# Phase 1 — Physics-loss multi-output neural network (Wells 4, 6, 7)

Train a **PyTorch multi-output regression network** to predict finalized composite curves from measured logs, marker identity/position and physics-derived features. The loss combines masked supervised errors with **differentiable physical penalties**. Leave-one-well-out (LOWO) validation, marker similarity review, checkshot-based TWT, and the verified datum offsets are retained.

**Run Cells 1–6D, then 7–11.** Install `torch` in your notebook kernel if needed. CPU by default; GPU automatically selected when available. Physics coefficients are provisional; no held-out well is used for early stopping. The final network is trained on all three wells and must be tested on a fourth independent well before operational use.


In [44]:
# Cell 1 — Imports
from pathlib import Path
import re, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lasio, joblib
from math import ceil
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.base import clone
from IPython.display import display
print('CPU multiwell workflow ready')
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import copy
print("PyTorch",torch.__version__,"CUDA available",torch.cuda.is_available())


CPU multiwell workflow ready
PyTorch 2.13.0+cu129 CUDA available False


In [45]:
# Cell 2 — Paths and editable settings
BASE=Path('/home/apgce_team_08/link-to-apgce_team_08')
WELL_DIRS={'Well_4':BASE/'Well_4','Well_6':BASE/'Well_6','Well_7':BASE/'Well_7'}
OUT=BASE/'Well_4_6_7_Phase1_Multiwell_CPU_Output'
OUT.mkdir(parents=True,exist_ok=True)
N_JOBS=8
# TRAINING OPTIMIZATION (EDIT HERE)
MAX_TRAIN_ROWS_PER_WELL=18000  # stratified depth sampling for CV; 0 = all
MAX_FINAL_ROWS_PER_WELL=60000  # larger refit for deployment; 0 = all
MAX_QC_ROWS_PER_WELL=12000     # evaluate representative held-out depth samples; 0 = all
USE_WELL_BALANCING=True        # equal effective contribution per training well
MODEL_SELECTION_METRIC="NRMSE" # per-well RMSE / (P95-P05); more comparable across wells

RANDOM_STATE=42
DEPTH_TOLERANCE_M=0.10
MIN_TRAIN_SAMPLES=150
MIN_TEST_SAMPLES=30
MIN_INPUT_CURVES=2
# Set filename overrides if automatic discovery is ambiguous (e.g. 'Well_6': {'raw':'...','composite':'...'}).
FILE_OVERRIDES={}
# Preferred measured curves from original Well 4; intersection of names across wells is used.
MEASURED_INPUTS=['CALI_P','DENS_P','DTC_P','GR_P','NEUT_P','NPHIHC','PEF_P','RHOBHC','RT_P']
TARGETS=['FCOAL','DTCOM','DTSH','PERMH','PHIE','PHIT','DENB','SWE','SWT','VCLB','VCLD','VCLW','VDOLO','VGAS','VLIME','VOIL','VSAND','VSHALE','VSHGR','VSILT']
# Optional curve name aliases (per-well). Set only after checking that units and meanings match.
CURVE_ALIASES={}  # e.g. {'Well_6': {'GR':'GR_P'}}
# EDIT HERE IN FUTURE: verified KB-to-sea-level datum offsets (metres).
# These values are intentionally hardcoded for ALL three wells and take precedence
# over marker-derived estimates. TVDSS_m = TVD_m - DATUM_OVERRIDES_M[well].
# Update a value here if the well datum is revised; then rerun Cell 2 onward.
DATUM_OVERRIDES_M={
    "Well_4": 21.978,
    "Well_6": 22.0,
    "Well_7": 29.0,
}
# Candidate comparison: nonlinear tree ensemble versus boosted trees.
# Increase estimator counts only after checking leave-one-well-out scores.
CANDIDATES=[
 {'name':'extra_fine','kind':'extra','n_estimators':350,'min_samples_leaf':2,'max_features':1.0},
 {'name':'extra_balanced','kind':'extra','n_estimators':450,'min_samples_leaf':5,'max_features':0.9},
 {'name':'extra_smooth','kind':'extra','n_estimators':350,'min_samples_leaf':12,'max_features':0.85},
 {'name':'hist_boost','kind':'hgb','max_iter':300,'learning_rate':0.055,'max_leaf_nodes':20,'l2_regularization':1.0},
]
print('Output:',OUT)
# MARKER-AWARE ML — EDIT HERE
USE_MARKER_FEATURES=True  # False: train using measured logs only
MARKER_FEATURES=['MARKER_DISTANCE_ABOVE_MS','MARKER_DISTANCE_BELOW_MS',
                 'MARKER_INTERVAL_FRACTION','MARKER_NEAREST_DISTANCE_MS']
# Markers must be supplied for a future well when USE_MARKER_FEATURES=True.

# NEAREST MARKER IDENTITY (EDIT HERE)
# One-hot columns are generated from the union of marker names in Wells 4/6/7.
# The names are normalized consistently (case/whitespace); unknown future markers
# map to all-zero identity columns, not to a fabricated training marker.
USE_NEAREST_MARKER_IDENTITY=True


# ============================================================
# PHYSICS-GUIDED FEATURES — EDIT HERE (PROVISIONAL DEFAULTS)
# ============================================================
# These are illustrative starting assumptions, NOT field-calibrated values.
# Recalibrate with core, water samples, logs, and local formation knowledge.
USE_PHYSICS_FEATURES=True
PHYSICS_PARAMS={
    'archie_a':1.0,           # tortuosity factor (dimensionless)
    'archie_m':2.0,           # cementation exponent
    'archie_n':2.0,           # saturation exponent
    'rw_ohm_m':0.10,          # formation-water resistivity at formation temperature
    'matrix_density_gcc':2.65, # quartz-rich sandstone hypothesis
    'fluid_density_gcc':1.00,  # water-filled pore hypothesis
    'matrix_sonic_usft':55.5,  # sandstone matrix transit time
    'fluid_sonic_usft':189.0,  # water transit time
    'resistivity_min_ohm_m':0.02,
    'porosity_floor_archie':0.02,
}
# Caution: Archie assumes clean, water-wet formation with suitable resistivity.
# Density and sonic porosity are proxies; coal, shale, gas and mixed lithology
# can violate these assumptions. No physics feature uses FINAL_ target curves.
# When deploying on another well, use the same parameters and feature code.
APPLY_PHYSICAL_BOUNDS_AT_INFERENCE=True
# PHIE<=PHIT is applied only when both targets are predicted; no mineral-volume
# closure or Archie hard constraint is imposed without field-specific calibration.

# ============================================================
# PHYSICS-LOSS NEURAL NETWORK — EDITABLE CONFIGURATION
# ============================================================
NN_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'  # use 'cpu' to force CPU
NN_HIDDEN = (192, 128, 96)
NN_DROPOUT = 0.12
NN_BATCH_SIZE = 1024
NN_EPOCHS = 90
NN_PATIENCE = 13
NN_LR = 0.001
NN_WEIGHT_DECAY = 0.0001
NN_VALIDATION_FRACTION = 0.15   # depth-blocked INTERNAL validation in training wells only
NN_MIN_TARGET_SAMPLES = 80
NN_MIN_TARGET_STD = 1e-4
NN_GRAD_CLIP = 2.0
# Penalties are in normalized-target units, using physical values for comparisons.
# Begin conservative; tune ONLY on training wells or nested validation.
LOSS_WEIGHTS = {
    'porosity_order': 0.12,    # PHIE <= PHIT
    'fraction_bounds': 0.08,  # PHIE/PHIT/SWE/... in [0,1]
    'permeability_positive': 0.03, # PERMH >= 0
    'density_proxy': 0.0,     # OFF: lithology/fluid-dependent; enable after calibration
    'archie_proxy': 0.0,      # OFF: clean-water-bearing-only, uncalibrated Rw
    'volume_closure': 0.0,    # OFF until disjoint volume curves are verified
}
# Optional conditional constraints: require explicit geological calibration.
DENSITY_PROXY_MAX_POROSITY = 0.45
ARCHIE_PROXY_MAX_SW = 1.0
# Only enable after verifying these curves are mutually exclusive and sum to one.
VOLUME_CLOSURE_TARGETS = []
# Optional user-approved canonical marker aliases. Review Cell 6C output first.
# E.g. {'TOP SAND A': 'SAND A TOP'}; empty means no automatic merging.
APPROVED_MARKER_MAPPING = {}


Output: /home/apgce_team_08/link-to-apgce_team_08/Well_4_6_7_Phase1_Multiwell_CPU_Output


In [46]:
# Cell 3 — Discover files (inspect printed mapping before proceeding)
def discover_files(well, folder):
    if not folder.is_dir(): raise FileNotFoundError(f'{well}: missing folder {folder}')
    override=FILE_OVERRIDES.get(well,{})
    las_files=sorted(folder.glob('*.las'))
    if 'raw' in override: raw=folder/override['raw']
    else:
        raw_candidates=[p for p in las_files if 'composite' not in p.name.lower() and 'interp' not in p.name.lower()]
        if len(raw_candidates)!=1: raise ValueError(f'{well}: ambiguous raw LAS: {[p.name for p in raw_candidates]}; set FILE_OVERRIDES')
        raw=raw_candidates[0]
    if 'composite' in override: composite=folder/override['composite']
    else:
        comp_candidates=[p for p in las_files if 'composite' in p.name.lower()]
        if len(comp_candidates)!=1: raise ValueError(f'{well}: ambiguous composite LAS: {[p.name for p in comp_candidates]}; set FILE_OVERRIDES')
        composite=comp_candidates[0]
    def txt_file(kind):
        if kind in override:return folder/override[kind]
        candidates=[p for p in folder.glob('*.txt') if kind.lower() in p.name.lower()]
        if kind == 'deviation' and len(candidates) == 0:
            print(f'{well}: no deviation file found; assuming vertical well (TVD = MDKB).')
            return None
        if len(candidates)!=1: raise ValueError(f'{well}: expected exactly one {kind} TXT, found {[p.name for p in candidates]}; set FILE_OVERRIDES')
        return candidates[0]
    paths={'raw':raw,'composite':composite,'checkshot':txt_file('checkshot'),
           'deviation':txt_file('deviation'),'markers':txt_file('markers')}
    for kind,path in paths.items():
        if path is None and kind == 'deviation': continue
        if not path.is_file():raise FileNotFoundError(f'{well} {kind}: {path}')
    return paths
FILES={w:discover_files(w,p) for w,p in WELL_DIRS.items()}
display(pd.DataFrame({w:{k:(str(v) if v is not None else 'VERTICAL ASSUMED') for k,v in p.items()} for w,p in FILES.items()}).T)

Well_7: no deviation file found; assuming vertical well (TVD = MDKB).


,raw,composite,checkshot,deviation,markers
Well_4,/home/apgce_team_08/link-to-apgce_team_08/Well...,/home/apgce_team_08/link-to-apgce_team_08/Well...,/home/apgce_team_08/link-to-apgce_team_08/Well...,/home/apgce_team_08/link-to-apgce_team_08/Well...,/home/apgce_team_08/link-to-apgce_team_08/Well...
Well_6,/home/apgce_team_08/link-to-apgce_team_08/Well...,/home/apgce_team_08/link-to-apgce_team_08/Well...,/home/apgce_team_08/link-to-apgce_team_08/Well...,/home/apgce_team_08/link-to-apgce_team_08/Well...,/home/apgce_team_08/link-to-apgce_team_08/Well...
Well_7,/home/apgce_team_08/link-to-apgce_team_08/Well...,/home/apgce_team_08/link-to-apgce_team_08/Well...,/home/apgce_team_08/link-to-apgce_team_08/Well...,VERTICAL ASSUMED,/home/apgce_team_08/link-to-apgce_team_08/Well...


In [47]:
# Cell 4 — LAS readers and checkshot-based time conversion (BEFORE registration / ML)
def las_frame(path, well):
    las=lasio.read(str(path),engine='normal')  # works with wrapped LAS
    df=las.df().reset_index()
    df.columns=[str(c).strip().upper() for c in df.columns]
    df=df.rename(columns={df.columns[0]:'MDKB_m'})
    for c in df.columns:
        df[c]=pd.to_numeric(df[c],errors='coerce').replace([np.inf,-np.inf,-9999,-999.25,9999],np.nan)
    df=df.dropna(subset=['MDKB_m']).sort_values('MDKB_m').drop_duplicates('MDKB_m').reset_index(drop=True)
    for old,new in CURVE_ALIASES.get(well,{}).items():
        old,new=old.upper(),new.upper()
        if old in df and new not in df: df=df.rename(columns={old:new})
    units={str(c.mnemonic).strip().upper():str(c.unit).strip() for c in las.curves}
    return df,units

def numeric_rows(path,ncols):
    rows=[]
    for line in Path(path).read_text().splitlines():
        parts=line.split()
        if len(parts)<ncols or line.lstrip().startswith('#'):continue
        try:rows.append([float(x) for x in parts[:ncols]])
        except ValueError:continue
    a=np.asarray(rows,dtype=float)
    if a.ndim!=2 or len(a)<2:raise ValueError(f'Invalid numeric table: {path}')
    a=a[np.argsort(a[:,0])]
    if np.any(np.diff(a[:,0])<=0):raise ValueError(f'Duplicate or non-monotonic first column: {path}')
    return a

def read_markers(path):
    """Parse marker NAME MDKB TVDSS or NAME TVDSS; preserve partial rows.

    Column ordering must be checked if a well uses a different convention.
    """
    rows=[]; skipped=[]
    for line_number,line in enumerate(Path(path).read_text(errors='replace').splitlines(),1):
        line=line.strip()
        if not line or line.startswith(('#','~',';')): continue
        parts=re.split(r'[\t,;]+|\s{2,}',line)
        parts=[x.strip().strip('"') for x in parts if x.strip()]
        if len(parts)<2:parts=line.split()
        if len(parts)<2:continue
        # Accept a marker name containing spaces if separated by whitespace.
        if len(parts)>3:
            parts=[' '.join(parts[:-2]),parts[-2],parts[-1]]
        name=parts[0]
        try:
            if len(parts)>=3:
                md=float(parts[1]);tvdss=float(parts[2])
            else:
                md=np.nan;tvdss=float(parts[1])
            if np.isfinite(tvdss):rows.append((name,md,tvdss))
        except (ValueError,TypeError):
            skipped.append((line_number,line[:100]))
    out=pd.DataFrame(rows,columns=['Marker','MDKB_m','Marker_TVDSS_m'])
    print(f'Marker parser {Path(path).name}: {len(out)} rows, {len(skipped)} unparsed lines')
    if skipped:print('  Unparsed examples:',skipped[:4])
    if out.empty:print('  WARNING: no marker rows parsed; inspect file layout and column order')
    return out

def time_converter(well,paths):
    vertical_assumed = paths['deviation'] is None
    dev = None if vertical_assumed else numeric_rows(paths['deviation'],4)
    shot=numeric_rows(paths['checkshot'],2)
    if np.any(np.diff(shot[:,1])<=0):raise ValueError(f'{well}: checkshot TWT must increase')
    markers=read_markers(paths['markers'])
    ties = markers.copy() if vertical_assumed else markers.loc[
        markers.MDKB_m.between(dev[0,0], dev[-1,0])
    ].copy()
    # Estimate the depth datum from available valid marker ties.
    # A single tie is permitted but cannot validate the datum consistency.
    ties=ties.dropna(subset=['MDKB_m','Marker_TVDSS_m'])
    if well in DATUM_OVERRIDES_M:
        datum=float(DATUM_OVERRIDES_M[well])
        datum_source='hardcoded override (edit DATUM_OVERRIDES_M in Cell 2)'
        if len(ties):
            tied_tvd=(ties.MDKB_m.to_numpy(dtype=float) if vertical_assumed else
                      np.interp(ties.MDKB_m,dev[:,0],dev[:,1]))
            residual=tied_tvd-ties.Marker_TVDSS_m.to_numpy(dtype=float)
            max_resid=float(np.max(np.abs(residual-datum)))
            if max_resid > 2:
                print(f'WARNING {well}: marker datum mismatch up to {max_resid:.2f} m; check hardcoded datum, markers and deviation.')
        else:
            max_resid=np.nan
    elif len(ties)>=1:
        tied_tvd=(ties.MDKB_m.to_numpy(dtype=float) if vertical_assumed else
                  np.interp(ties.MDKB_m,dev[:,0],dev[:,1]))
        residual=tied_tvd-ties.Marker_TVDSS_m.to_numpy(dtype=float)
        datum=float(np.median(residual))
        max_resid=float(np.max(np.abs(residual-datum)))
        datum_source=f'{len(ties)} marker tie(s)'
        if max_resid>2:
            raise ValueError(f'{well}: datum inconsistency {max_resid:.2f}m; inspect deviation and marker datums')
        if len(ties)==1:
            print(f'WARNING {well}: datum inferred from ONE marker tie only; verify against LAS header / well report.')
    else:
        # Inspect LAS headers; do not assume KB or zero datum without verification.
        header_candidates=[]
        for las_path in (paths['raw'],paths['composite']):
            las=lasio.read(str(las_path),engine='normal')
            for key in ('EKB','KB','KBE','KBEL','RKB','EREF'):
                try:
                    item=las.well[key]
                    value=float(item.value)
                    if np.isfinite(value):
                        header_candidates.append((las_path.name,key,value))
                except (KeyError,TypeError,ValueError,AttributeError):
                    pass
        print(f'{well}: no usable marker MDKB/TVDSS ties.')
        print(f'{well}: possible LAS elevation metadata (NOT automatically applied): {header_candidates}')
        raise ValueError(
            f'{well}: depth datum is unverified. Set DATUM_OVERRIDES_M["{well}"] '
            'to a verified KB-to-sea-level offset in metres, then rerun Cell 4. '
            'Do not use zero unless the well datum is confirmed as sea level.'
        )
    if not np.isfinite(datum):
        raise ValueError(f'{well}: non-finite datum')
    print(f'{well}: datum source: {datum_source}')
    def convert(df):
        d=df.copy()
        md=d.MDKB_m.to_numpy(dtype=float)
        d['TVD_m']=(md.copy() if vertical_assumed else
                    np.interp(md,dev[:,0],dev[:,1],left=np.nan,right=np.nan))
        d['TVDSS_m']=d.TVD_m-datum
        d['TWT_ms']=np.interp(d.TVDSS_m.to_numpy(dtype=float),shot[:,0],shot[:,1],left=np.nan,right=np.nan)
        return d
    markers['TWT_ms']=np.interp(markers.Marker_TVDSS_m.to_numpy(dtype=float),shot[:,0],shot[:,1],left=np.nan,right=np.nan)
    if vertical_assumed:
        print(f'{well}: VERTICAL-WELL ASSUMPTION active (TVD=MDKB); verify if deviation survey becomes available.')
    print(f'{well}: datum={datum:.3f}m; marker residual max={max_resid:.3f}m; checkshot {shot[0,1]:.1f}–{shot[-1,1]:.1f}ms')
    good_markers=markers['TWT_ms'].notna().sum()
    print(f'{well}: markers parsed={len(markers)}, within checkshot TWT range={good_markers}')
    if len(markers) and not good_markers:
        print(f'WARNING {well}: all marker TWT are NaN. Check marker TVDSS values, column order and checkshot depth range.')
    elif good_markers:
        print(f'{well}: marker TWT span {markers.TWT_ms.min():.1f}–{markers.TWT_ms.max():.1f} ms')
    return convert,markers,datum

WELLS={}
for well,paths in FILES.items():
    raw,raw_units=las_frame(paths['raw'],well)
    comp,comp_units=las_frame(paths['composite'],well)
    convert,markers,datum=time_converter(well,paths)
    raw=convert(raw);comp=convert(comp)
    WELLS[well]={'raw':raw,'comp':comp,'raw_units':raw_units,'comp_units':comp_units,'markers':markers,'datum_m':datum}
    print(f'{well}: raw {len(raw):,} samples; composite {len(comp):,}; valid composite TWT {comp.TWT_ms.notna().sum():,}')

Marker parser Well-4_Markers.txt: 22 rows, 0 unparsed lines
Well_4: datum source: hardcoded override (edit DATUM_OVERRIDES_M in Cell 2)
Well_4: datum=21.978m; marker residual max=0.040m; checkshot 1151.8–2158.4ms
Well_4: markers parsed=22, within checkshot TWT range=21
Well_4: marker TWT span 1246.6–2135.3 ms
Well_4: raw 10,364 samples; composite 10,364; valid composite TWT 9,824
Marker parser Well-6_Markers.txt: 15 rows, 0 unparsed lines
Well_6: datum source: hardcoded override (edit DATUM_OVERRIDES_M in Cell 2)
Well_6: datum=22.000m; marker residual max=nanm; checkshot 564.4–1910.0ms
Well_6: markers parsed=15, within checkshot TWT range=15
Well_6: marker TWT span 1493.1–1864.7 ms
Well_6: raw 7,975 samples; composite 7,802; valid composite TWT 7,763
Marker parser Well-7_Markers.txt: 7 rows, 0 unparsed lines
Well_7: datum source: hardcoded override (edit DATUM_OVERRIDES_M in Cell 2)
Well_7: VERTICAL-WELL ASSUMPTION active (TVD=MDKB); verify if deviation survey becomes available.
Well_7

In [48]:
# Cell 5 — Inspect curve availability and units across wells BEFORE ML
availability=[]
for curve in MEASURED_INPUTS:
    availability.append({'Curve':curve,'Role':'measured',**{w:('YES / '+d['raw_units'].get(curve,'?')) if curve in d['raw'] else 'MISSING' for w,d in WELLS.items()}})
for curve in TARGETS:
    availability.append({'Curve':curve,'Role':'composite target',**{w:('YES / '+d['comp_units'].get(curve,'?')) if curve in d['comp'] else 'MISSING' for w,d in WELLS.items()}})
availability_df=pd.DataFrame(availability)
display(availability_df)
# Shared inputs enforce comparable feature definitions for all wells. Missing values within a curve are imputed.
INPUTS=[c for c in MEASURED_INPUTS if all(c in d['raw'] for d in WELLS.values())]
OUTPUTS=[c for c in TARGETS if any(c in d['comp'] for d in WELLS.values())]
if len(INPUTS)<MIN_INPUT_CURVES:raise RuntimeError(f'Only {len(INPUTS)} common measured curves. Review aliases/units; do not silently mix different logs.')
# Confirmed by interpreter: unit scales are consistent in Wells 4/6/7.
# Empty LAS unit fields are metadata omissions, not numerical conversions.
def canonical_unit(unit):
    u=str(unit or '').strip().upper().replace(' ', '')
    return {'G/C3':'G/CC','G/CM3':'G/CC','G/CM^3':'G/CC'}.get(u,u)
for role,curves,field in [('Measured',INPUTS,'raw_units'),('Target',OUTPUTS,'comp_units')]:
    for c in curves:
        specified={w:canonical_unit(d[field].get(c,'')) for w,d in WELLS.items()
                   if c in d['raw' if role=='Measured' else 'comp']}
        nonblank={u for u in specified.values() if u}
        if len(nonblank)>1:
            warnings.warn(f'{role} {c}: conflicting EXPLICIT units {specified}; verify before training')
print('LAS blank units accepted for verified Wells 4/6/7; numerical values unchanged.')
print('Common measured inputs:',INPUTS)
print('Composite targets available:',OUTPUTS)
# IMPORTANT: stop here to inspect unit differences and curve meanings before proceeding.

,Curve,Role,Well_4,Well_6,Well_7
0,CALI_P,measured,YES / in,YES / in,YES / in
1,DENS_P,measured,YES / g/cm3,YES / g/cm3,YES / g/cm3
2,DTC_P,measured,YES / us/ft,MISSING,MISSING
3,GR_P,measured,YES / gAPI,YES / gAPI,YES / gAPI
4,NEUT_P,measured,YES / frac,YES / frac,YES / frac
5,NPHIHC,measured,YES / frac,YES / frac,YES / frac
6,PEF_P,measured,YES / b/e,YES / b/e,YES / b/e
7,RHOBHC,measured,YES / g/cm3,YES / g/cm3,YES / g/cm3
8,RT_P,measured,YES / ohm.m,YES / ohm.m,YES / ohm.m
9,FCOAL,composite target,YES / V/V,YES /,YES /


LAS blank units accepted for verified Wells 4/6/7; numerical values unchanged.
Common measured inputs: ['CALI_P', 'DENS_P', 'GR_P', 'NEUT_P', 'NPHIHC', 'PEF_P', 'RHOBHC', 'RT_P']
Composite targets available: ['FCOAL', 'DTCOM', 'DTSH', 'PERMH', 'PHIE', 'PHIT', 'DENB', 'SWE', 'SWT', 'VCLB', 'VCLD', 'VCLW', 'VDOLO', 'VGAS', 'VLIME', 'VOIL', 'VSAND', 'VSHALE', 'VSHGR', 'VSILT']


In [49]:
# Cell 6 — Align each well by measured depth; no row-number alignment
JOINED={}
for well,d in WELLS.items():
    raw=d['raw']; comp=d['comp']; available=[c for c in OUTPUTS if c in comp]
    left=comp[['MDKB_m','TVD_m','TVDSS_m','TWT_ms']+available].copy()
    right=raw[['MDKB_m']+INPUTS].copy()
    # Explicit prefix prevents target/input collision.
    left=left.rename(columns={c:'FINAL_'+c for c in available})
    right=right.rename(columns={c:'RAW_'+c for c in INPUTS})
    joined=pd.merge_asof(left.sort_values('MDKB_m'),right.sort_values('MDKB_m'),on='MDKB_m',direction='nearest',tolerance=DEPTH_TOLERANCE_M)
    joined.insert(0,'Well',well)
    for c in OUTPUTS:
        if 'FINAL_'+c not in joined:joined['FINAL_'+c]=np.nan
    JOINED[well]=joined.reset_index(drop=True)
    print(f'{well}: matched rows with >=1 input: {joined[["RAW_"+c for c in INPUTS]].notna().any(axis=1).sum():,}; TWT: {joined.TWT_ms.notna().sum():,}')
DATA=pd.concat(JOINED.values(),ignore_index=True)
DATA.to_csv(OUT/'Well4_6_7_Aligned_Training_Data.csv',index=False)
print('Total samples:',len(DATA))

Well_4: matched rows with >=1 input: 10,364; TWT: 9,824
Well_6: matched rows with >=1 input: 7,802; TWT: 7,763
Well_7: matched rows with >=1 input: 7,419; TWT: 6,927
Total samples: 25585


In [50]:
# Cell 6B — Marker distances + nearest marker identity (run BEFORE Cell 7)
# The nearest marker's NAME is encoded as one-hot numeric features,
# allowing tree models to learn marker-specific stratigraphic responses.
# Marker names must correspond across wells; standardize aliases before training.
# Do not use petrophysical composite curves or labels to construct these features.

import re

def normalize_marker_name(name):
    normalized=re.sub(r"\s+", " ", str(name).strip().upper())
    return APPROVED_MARKER_MAPPING.get(normalized, normalized)

def marker_vocabulary(wells):
    names=set()
    for d in wells.values():
        markers=d['markers']
        if markers is None or not {'Marker','TWT_ms'}.issubset(markers.columns):
            continue
        for name in markers.loc[pd.to_numeric(markers['TWT_ms'],errors='coerce').notna(),'Marker']:
            name=normalize_marker_name(name)
            if name and name not in ('NAN','NONE'):
                names.add(name)
    return sorted(names)

MARKER_VOCABULARY = marker_vocabulary(WELLS) if USE_NEAREST_MARKER_IDENTITY else []
MARKER_ID_COLUMNS = [f'MARKER_NEAREST_ID_{i:03d}' for i in range(len(MARKER_VOCABULARY))]
MARKER_ID_LOOKUP = dict(zip(MARKER_VOCABULARY,MARKER_ID_COLUMNS))
MARKER_FEATURES = ['MARKER_DISTANCE_ABOVE_MS','MARKER_DISTANCE_BELOW_MS',
                   'MARKER_INTERVAL_FRACTION','MARKER_NEAREST_DISTANCE_MS'] + MARKER_ID_COLUMNS

def marker_features(twt_values, markers, vocabulary=None):
    """Build numerical stratigraphic features plus nearest marker identity.

    Unknown marker names receive all-zero one-hot columns; missing marker
    times receive NaN distance features. This is a distinct 'unknown' state
    only when accompanied by the marker-availability indicator.
    """
    vocab = MARKER_VOCABULARY if vocabulary is None else list(vocabulary)
    id_cols=[f'MARKER_NEAREST_ID_{i:03d}' for i in range(len(vocab))]
    dist_cols=['MARKER_DISTANCE_ABOVE_MS','MARKER_DISTANCE_BELOW_MS',
               'MARKER_INTERVAL_FRACTION','MARKER_NEAREST_DISTANCE_MS']
    twt=np.asarray(twt_values,dtype=float)
    result=pd.DataFrame(np.nan,index=np.arange(len(twt)),columns=dist_cols,dtype=float)
    for col in id_cols: result[col]=0.0
    result['MARKER_NEAREST_NAME']=''
    result['MARKER_ID_KNOWN']=0.0
    if markers is None or not {'TWT_ms','Marker'}.issubset(markers.columns):
        return result
    mm=markers[['Marker','TWT_ms']].copy()
    mm['TWT_ms']=pd.to_numeric(mm['TWT_ms'],errors='coerce')
    mm=mm[np.isfinite(mm['TWT_ms'])].sort_values('TWT_ms',kind='stable')
    if mm.empty:return result
    # Keep one name per marker time; stable tie-breaking if duplicates exist.
    mm=mm.drop_duplicates('TWT_ms',keep='first')
    mt=mm['TWT_ms'].to_numpy(dtype=float)
    names=np.array([normalize_marker_name(v) for v in mm['Marker']],dtype=object)
    valid=np.isfinite(twt)
    if not valid.any():return result
    y=twt[valid]
    pos=np.searchsorted(mt,y,side='right')
    above=np.full(len(y),np.nan);below=np.full(len(y),np.nan)
    has_above=pos>0;has_below=pos<len(mt)
    above[has_above]=y[has_above]-mt[pos[has_above]-1]
    below[has_below]=mt[pos[has_below]]-y[has_below]
    both=has_above & has_below
    frac=np.full(len(y),np.nan)
    span=above[both]+below[both]
    frac[both]=np.divide(above[both],span,out=np.full(len(span),np.nan),where=span>1e-8)
    # The nearest marker may lie above or below the sample.
    # On equal distances choose the shallower marker.
    left_idx=np.clip(pos-1,0,len(mt)-1)
    right_idx=np.clip(pos,0,len(mt)-1)
    left_d=np.abs(y-mt[left_idx]);right_d=np.abs(y-mt[right_idx])
    nearest_idx=np.where(left_d<=right_d,left_idx,right_idx)
    nearest_names=names[nearest_idx]
    result.loc[valid,'MARKER_DISTANCE_ABOVE_MS']=above
    result.loc[valid,'MARKER_DISTANCE_BELOW_MS']=below
    result.loc[valid,'MARKER_INTERVAL_FRACTION']=frac
    result.loc[valid,'MARKER_NEAREST_DISTANCE_MS']=np.minimum(left_d,right_d)
    result.loc[valid,'MARKER_NEAREST_NAME']=nearest_names
    lookup=dict(zip(vocab,id_cols))
    valid_indices=np.flatnonzero(valid)
    for name in np.unique(nearest_names):
        if name in lookup:
            idx=valid_indices[nearest_names==name]
            result.loc[idx,lookup[name]]=1.0
            result.loc[idx,'MARKER_ID_KNOWN']=1.0
    return result

for well in WELLS:
    frame=JOINED[well]
    extra=marker_features(frame['TWT_ms'],WELLS[well]['markers'],MARKER_VOCABULARY)
    for col in MARKER_FEATURES:frame[col]=extra[col].to_numpy()
    frame['MARKER_NEAREST_NAME']=extra['MARKER_NEAREST_NAME'].to_numpy() # QC only, NOT an ML input
    n_known=int(extra['MARKER_ID_KNOWN'].sum())
    print(f'{well}: {n_known:,}/{len(frame):,} samples with a known nearest marker identity; '
          f'{extra.loc[extra.MARKER_NEAREST_NAME.ne(""),"MARKER_NEAREST_NAME"].nunique()} unique nearest markers')
DATA=pd.concat(JOINED.values(),ignore_index=True)
print('Marker identity vocabulary:',dict(enumerate(MARKER_VOCABULARY)))
print('Marker-aware features:',len(MARKER_FEATURES))
print('Rerun Cell 7 onward to retrain with nearest marker identity.')


Well_4: 9,824/10,364 samples with a known nearest marker identity; 21 unique nearest markers
Well_6: 7,763/7,802 samples with a known nearest marker identity; 15 unique nearest markers
Well_7: 6,927/7,419 samples with a known nearest marker identity; 7 unique nearest markers
Marker identity vocabulary: {0: 'GROUP_H', 1: 'GROUP_I', 2: 'I-1', 3: 'I-100', 4: 'I-115', 5: 'I-118', 6: 'I-120', 7: 'I-25', 8: 'I-35', 9: 'I-35_LOWER', 10: 'I-35_UPPER', 11: 'I-50_COAL', 12: 'I-68', 13: 'I-70', 14: 'I-75', 15: 'I-85', 16: 'I-95', 17: 'J-10', 18: 'J-15', 19: 'J-20', 20: 'K-20', 21: 'K-22', 22: 'K-25_LOWER', 23: 'K-25_UPPER', 24: 'K-28', 25: 'K-30', 26: 'TOP_GROUP_H', 27: 'TOP_GROUP_I', 28: 'TOP_GROUP_J', 29: 'TOP_GROUP_K'}
Marker-aware features: 34
Rerun Cell 7 onward to retrain with nearest marker identity.


In [51]:
# Cell 6C — Flag SIMILAR marker names for future geological correlation / merging
# IMPORTANT: This is a REVIEW FLAG only. It NEVER automatically merges markers,
# changes their identities, or alters ML features/predictions.
# Run after Cell 6B. Edit thresholds below if desired.

from difflib import SequenceMatcher
from itertools import combinations

MARKER_SIMILARITY_THRESHOLD = 0.78  # EDIT HERE: 0–1; lower = more candidate matches
MARKER_IGNORE_TOP_SUFFIX = True     # Treat "Sand A Top" and "Sand A" as possible matches
MARKER_IGNORE_PUNCTUATION = True
MARKER_REQUIRE_DIFFERENT_WELLS = False  # True: only flag names occurring in different wells


def marker_comparison_key(name):
    s = normalize_marker_name(name)
    if MARKER_IGNORE_PUNCTUATION:
        s = re.sub(r"[^A-Z0-9]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    if MARKER_IGNORE_TOP_SUFFIX:
        # Remove standalone TOP at either end; keep original marker names unchanged.
        s = re.sub(r"^(?:TOP)\s+|\s+(?:TOP)$", "", s).strip()
    return s


marker_rows = []
for well, well_data in WELLS.items():
    markers = well_data['markers']
    if markers is None or 'Marker' not in markers.columns:
        continue
    for _, m in markers.iterrows():
        original = str(m['Marker']).strip()
        if not original or original.upper() in ('NAN', 'NONE'):
            continue
        marker_rows.append({
            'Well': well,
            'Original_Marker': original,
            'Normalized_Marker': normalize_marker_name(original),
            'Comparison_Key': marker_comparison_key(original),
            'TWT_ms': pd.to_numeric(m.get('TWT_ms', np.nan), errors='coerce'),
        })
MARKER_INVENTORY = pd.DataFrame(marker_rows, columns=[
    'Well', 'Original_Marker', 'Normalized_Marker', 'Comparison_Key', 'TWT_ms'
])

# Compare distinct names, not every depth sample, to keep this lightweight.
unique_names = sorted(MARKER_INVENTORY['Normalized_Marker'].unique())
name_wells = {name: set(MARKER_INVENTORY.loc[
    MARKER_INVENTORY['Normalized_Marker'].eq(name), 'Well']) for name in unique_names}
comparison = {name: marker_comparison_key(name) for name in unique_names}

candidate_rows = []
for a, b in combinations(unique_names, 2):
    if MARKER_REQUIRE_DIFFERENT_WELLS and not (
        name_wells[a] - name_wells[b] or name_wells[b] - name_wells[a]
    ):
        continue
    ka, kb = comparison[a], comparison[b]
    if not ka or not kb:
        continue
    similarity = SequenceMatcher(None, ka, kb).ratio()
    if similarity < MARKER_SIMILARITY_THRESHOLD:
        continue
    candidate_rows.append({
        'Marker_A': a,
        'Marker_B': b,
        'Wells_A': ', '.join(sorted(name_wells[a])),
        'Wells_B': ', '.join(sorted(name_wells[b])),
        'Comparison_Key_A': ka,
        'Comparison_Key_B': kb,
        'Name_Similarity': round(similarity, 4),
        'Exact_After_Normalization': ka == kb,
        'Review_Status': 'REVIEW',  # Edit this later to APPROVED or REJECTED
        'Canonical_Marker_Name': '',  # Fill only after geological confirmation
        'Interpreter_Notes': '',
    })

MARKER_SIMILARITY_REVIEW = pd.DataFrame(candidate_rows, columns=[
    'Marker_A', 'Marker_B', 'Wells_A', 'Wells_B',
    'Comparison_Key_A', 'Comparison_Key_B', 'Name_Similarity',
    'Exact_After_Normalization', 'Review_Status',
    'Canonical_Marker_Name', 'Interpreter_Notes'
]).sort_values('Name_Similarity', ascending=False).reset_index(drop=True)

# Flag names in the inventory and annotate the nearest marker for QC only.
flagged_names = set(MARKER_SIMILARITY_REVIEW['Marker_A']) | set(MARKER_SIMILARITY_REVIEW['Marker_B'])
MARKER_INVENTORY['Similar_Name_Flag'] = MARKER_INVENTORY['Normalized_Marker'].isin(flagged_names)
for well, frame in JOINED.items():
    if 'MARKER_NEAREST_NAME' in frame.columns:
        frame['MARKER_NEAREST_SIMILAR_NAME_FLAG'] = (
            frame['MARKER_NEAREST_NAME'].isin(flagged_names).astype('int8')
        )  # QC-only flag; deliberately NOT appended to FEATURES

OUT.mkdir(parents=True, exist_ok=True)
inventory_path = OUT / 'Phase1_Marker_Name_Inventory.csv'
review_path = OUT / 'Phase1_Similar_Marker_Names_Review.csv'
MARKER_INVENTORY.to_csv(inventory_path, index=False)
MARKER_SIMILARITY_REVIEW.to_csv(review_path, index=False)

print(f'Marker names: {len(unique_names)}; candidate similar-name pairs: {len(MARKER_SIMILARITY_REVIEW)}')
print('Inventory:', inventory_path)
print('Review table:', review_path)
print('No marker merging or ML feature changes were applied.')
display(MARKER_SIMILARITY_REVIEW.head(100))


Marker names: 31; candidate similar-name pairs: 19
Inventory: /home/apgce_team_08/link-to-apgce_team_08/Well_4_6_7_Phase1_Multiwell_CPU_Output/Phase1_Marker_Name_Inventory.csv
Review table: /home/apgce_team_08/link-to-apgce_team_08/Well_4_6_7_Phase1_Multiwell_CPU_Output/Phase1_Similar_Marker_Names_Review.csv
No marker merging or ML feature changes were applied.


,Marker_A,Marker_B,Wells_A,Wells_B,Comparison_Key_A,Comparison_Key_B,Name_Similarity,Exact_After_Normalization,Review_Status,Canonical_Marker_Name,Interpreter_Notes
0,GROUP_H,TOP_GROUP_H,Well_7,Well_4,GROUP H,GROUP H,1.0000,True,REVIEW,,
1,GROUP_I,TOP_GROUP_I,Well_7,Well_4,GROUP I,GROUP I,1.0000,True,REVIEW,,
2,GROUP_H,GROUP_I,Well_7,Well_7,GROUP H,GROUP I,0.8571,False,REVIEW,,
3,GROUP_H,TOP_GROUP_I,Well_7,Well_4,GROUP H,GROUP I,0.8571,False,REVIEW,,
4,GROUP_H,TOP_GROUP_J,Well_7,Well_4,GROUP H,GROUP J,0.8571,False,REVIEW,,
5,GROUP_H,TOP_GROUP_K,Well_7,Well_4,GROUP H,GROUP K,0.8571,False,REVIEW,,
6,GROUP_I,TOP_GROUP_H,Well_7,Well_4,GROUP I,GROUP H,0.8571,False,REVIEW,,
7,GROUP_I,TOP_GROUP_J,Well_7,Well_4,GROUP I,GROUP J,0.8571,False,REVIEW,,
8,GROUP_I,TOP_GROUP_K,Well_7,Well_4,GROUP I,GROUP K,0.8571,False,REVIEW,,
9,TOP_GROUP_H,TOP_GROUP_I,Well_4,Well_4,GROUP H,GROUP I,0.8571,False,REVIEW,,


In [52]:
# Cell 6D — Physics-derived features (measured curves only; no target leakage)
# Run AFTER Cell 6B/6C and BEFORE Cell 7.
# Parameter values are editable in Cell 2. Values are uncalibrated hypotheses.

PHYSICS_FEATURES = [
    'PHYS_DENSITY_POROSITY', 'PHYS_SONIC_POROSITY',
    'PHYS_LOG10_RT', 'PHYS_ARCHIE_SW_PROXY',
    'PHYS_DENSITY_VALID', 'PHYS_SONIC_VALID', 'PHYS_ARCHIE_VALID'
]

def physics_features(raw_frame, params=None):
    p=PHYSICS_PARAMS if params is None else params
    result=pd.DataFrame(index=raw_frame.index)
    def measured(curve):
        col='RAW_'+curve
        if col not in raw_frame.columns:
            return pd.Series(np.nan,index=raw_frame.index,dtype=float)
        return pd.to_numeric(raw_frame[col],errors='coerce').replace([np.inf,-np.inf],np.nan)
    rho=measured('DENS_P')
    # Only use RHOBHC if DENS_P is absent; do not silently mix both definitions.
    if 'RAW_DENS_P' not in raw_frame.columns:
        rho=measured('RHOBHC')
    dt=measured('DTC_P')
    rt=measured('RT_P')
    rho_ma=float(p['matrix_density_gcc']);rho_f=float(p['fluid_density_gcc'])
    dt_ma=float(p['matrix_sonic_usft']);dt_f=float(p['fluid_sonic_usft'])
    if abs(rho_ma-rho_f)<1e-8 or abs(dt_f-dt_ma)<1e-8:
        raise ValueError('Invalid matrix/fluid endmembers in PHYSICS_PARAMS')
    # No clipping: out-of-range estimates are diagnostic of model mismatch.
    phi_d=(rho_ma-rho)/(rho_ma-rho_f)
    phi_s=(dt-dt_ma)/(dt_f-dt_ma)
    good_d=phi_d.between(0.0,0.65)
    good_s=phi_s.between(0.0,0.65)
    result['PHYS_DENSITY_POROSITY']=phi_d.where(good_d)
    result['PHYS_SONIC_POROSITY']=phi_s.where(good_s)
    good_rt=rt>float(p['resistivity_min_ohm_m'])
    result['PHYS_LOG10_RT']=np.log10(rt.where(good_rt))
    # Archie SW is a *proxy* based on density-derived phi, NOT a truth label.
    # Do not use as hard saturation constraint in shaly/coal/gas intervals.
    good_archie=good_d & (phi_d>=float(p['porosity_floor_archie'])) & good_rt
    a,m,n,rw=(float(p[k]) for k in ('archie_a','archie_m','archie_n','rw_ohm_m'))
    if min(a,m,n,rw)<=0:raise ValueError('Archie parameters must be positive')
    with np.errstate(divide='ignore',invalid='ignore',over='ignore'):
        sw=(a*rw/(phi_d.pow(m)*rt)).pow(1.0/n)
    result['PHYS_ARCHIE_SW_PROXY']=sw.where(good_archie & np.isfinite(sw))
    result['PHYS_DENSITY_VALID']=good_d.astype(float)
    result['PHYS_SONIC_VALID']=good_s.astype(float)
    result['PHYS_ARCHIE_VALID']=good_archie.astype(float)
    return result

for well, frame in JOINED.items():
    extra=physics_features(frame)
    for col in PHYSICS_FEATURES:
        frame[col]=extra[col].to_numpy()
    print(well, '| density proxy valid:',int(extra.PHYS_DENSITY_VALID.sum()),
          '| Archie proxy valid:',int(extra.PHYS_ARCHIE_VALID.sum()))
DATA=pd.concat(JOINED.values(),ignore_index=True)
print('Physics features:',PHYSICS_FEATURES)
print('Physics parameters (provisional):',PHYSICS_PARAMS)


Well_4 | density proxy valid: 9509 | Archie proxy valid: 8548
Well_6 | density proxy valid: 7779 | Archie proxy valid: 7753
Well_7 | density proxy valid: 7362 | Archie proxy valid: 7352
Physics features: ['PHYS_DENSITY_POROSITY', 'PHYS_SONIC_POROSITY', 'PHYS_LOG10_RT', 'PHYS_ARCHIE_SW_PROXY', 'PHYS_DENSITY_VALID', 'PHYS_SONIC_VALID', 'PHYS_ARCHIE_VALID']
Physics parameters (provisional): {'archie_a': 1.0, 'archie_m': 2.0, 'archie_n': 2.0, 'rw_ohm_m': 0.1, 'matrix_density_gcc': 2.65, 'fluid_density_gcc': 1.0, 'matrix_sonic_usft': 55.5, 'fluid_sonic_usft': 189.0, 'resistivity_min_ohm_m': 0.02, 'porosity_floor_archie': 0.02}


## Cell 7 — Physics-loss neural network: LOWO validation

Multi-output network with **masked supervised loss** (targets missing in a well do not contribute), plus optional physics penalties. Training preprocessing (median imputation and feature/target scaling) is fitted **only on the two training wells** for each fold. Early stopping uses a depth-blocked internal validation subset from those training wells, never the held-out well. The physical bounds are not applied to the validation predictions: reported errors reflect the actual network output.

In [53]:
# Cell 7A — Helpers, model and differentiable physics loss
import os, random
random.seed(RANDOM_STATE); np.random.seed(RANDOM_STATE); torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(RANDOM_STATE)
DEVICE = torch.device(NN_DEVICE)
FEATURES = ['RAW_'+c for c in INPUTS] + (MARKER_FEATURES if USE_MARKER_FEATURES else []) + (PHYSICS_FEATURES if USE_PHYSICS_FEATURES else [])
TARGET_LIST = list(OUTPUTS)
# The validity test deliberately requires measured input(s), not marker/physics features alone.
def valid_inputs(df):
    return df[['RAW_'+c for c in INPUTS]].notna().any(axis=1)
def select_depth_samples(df, cap):
    if not cap or len(df)<=cap: return df.copy()
    return df.iloc[np.unique(np.linspace(0,len(df)-1,int(cap),dtype=int))].copy()

class PhysicsNet(nn.Module):
    def __init__(self, n_features, n_targets, hidden=NN_HIDDEN, dropout=NN_DROPOUT):
        super().__init__()
        layers=[]; last=n_features
        for h in hidden:
            layers += [nn.Linear(last,h),nn.LayerNorm(h),nn.GELU(),nn.Dropout(dropout)]
            last=h
        layers += [nn.Linear(last,n_targets)]
        self.network=nn.Sequential(*layers)
    def forward(self,x): return self.network(x)

FRACTION_TARGETS = ['PHIE','PHIT','SWE','SWT','FCOAL','VGAS','VOIL','VSAND','VSILT','VSHALE','VCLW','VDOLO','VLIME','VCLB','VCLD','VSHGR']

def prepare_training(train_df):
    # Fit imputer and scaler on training wells only.
    xraw=train_df[FEATURES].replace([np.inf,-np.inf],np.nan)
    # All-empty features are retained as zeros, allowing stable inference dimensions.
    imputer=SimpleImputer(strategy='median',keep_empty_features=True)
    ximp=imputer.fit_transform(xraw)
    scaler=StandardScaler().fit(ximp)
    xx=scaler.transform(ximp).astype('float32')
    y=train_df[['FINAL_'+t for t in TARGET_LIST]].apply(pd.to_numeric,errors='coerce').to_numpy(dtype=np.float64,copy=True)
    y=np.array(y,dtype=np.float64,copy=True)  # writable, independent of pandas backing memory
    y[~np.isfinite(y)]=np.nan
    counts=np.isfinite(y).sum(axis=0)
    active=counts>=NN_MIN_TARGET_SAMPLES
    means=np.array([np.nanmean(y[:,j]) if active[j] else 0.0 for j in range(y.shape[1])])
    stds=np.array([max(float(np.nanstd(y[:,j])),NN_MIN_TARGET_STD) if active[j] else 1.0 for j in range(y.shape[1])])
    yn=(y-means)/stds
    mask=np.isfinite(yn) & active[None,:]
    yn=np.where(mask,yn,0).astype('float32')
    return {'imputer':imputer,'scaler':scaler,'means':means,'stds':stds,'active':active},xx,yn,mask.astype('float32')

def transform_x(df, prep):
    xraw=df[FEATURES].replace([np.inf,-np.inf],np.nan)
    return prep['scaler'].transform(prep['imputer'].transform(xraw)).astype('float32')

def physics_loss(pred_norm, prep, raw_batch=None):
    # Predicted physical units, gradients flow back through scaling.
    mean=torch.as_tensor(prep['means'],dtype=pred_norm.dtype,device=pred_norm.device)
    std=torch.as_tensor(prep['stds'],dtype=pred_norm.dtype,device=pred_norm.device)
    active=prep['active']; p=pred_norm*std+mean
    idx={t:i for i,t in enumerate(TARGET_LIST)}
    terms={}; zero=p.sum()*0.0
    if all(t in idx and active[idx[t]] for t in ('PHIE','PHIT')):
        # Relative to target standard deviation to keep penalties scale-independent.
        a,b=idx['PHIE'],idx['PHIT']
        terms['porosity_order']=(torch.relu((p[:,a]-p[:,b])/max(stdsafe(prep,a),stdsafe(prep,b)))**2).mean()
    else: terms['porosity_order']=zero
    bounds=[]
    for t in FRACTION_TARGETS:
        if t in idx and active[idx[t]]:
            j=idx[t]; bounds.append((torch.relu(-p[:,j])**2+torch.relu(p[:,j]-1)**2).mean())
    terms['fraction_bounds']=torch.stack(bounds).mean() if bounds else zero
    terms['permeability_positive']=(torch.relu(-p[:,idx['PERMH']])**2).mean() if 'PERMH' in idx and active[idx['PERMH']] else zero
    # Proxies require masks derived from measured logs, not target composite values.
    # Disabled by default because sandstone/clean-formation assumptions do not hold everywhere.
    if raw_batch is not None and 'PHIE' in idx and active[idx['PHIE']] and 'PHYS_DENSITY_POROSITY' in raw_batch:
        v=raw_batch['PHYS_DENSITY_POROSITY']; good=torch.isfinite(v)&(v>=0)&(v<=DENSITY_PROXY_MAX_POROSITY)
        terms['density_proxy']=((p[good,idx['PHIE']]-v[good])**2).mean() if good.any() else zero
    else: terms['density_proxy']=zero
    if raw_batch is not None and 'SWE' in idx and active[idx['SWE']] and 'PHYS_ARCHIE_SW_PROXY' in raw_batch:
        v=raw_batch['PHYS_ARCHIE_SW_PROXY']; good=torch.isfinite(v)&(v>=0)&(v<=ARCHIE_PROXY_MAX_SW)
        # An additional clean-formation mask is REQUIRED before activating this term.
        terms['archie_proxy']=((p[good,idx['SWE']]-v[good])**2).mean() if good.any() else zero
    else: terms['archie_proxy']=zero
    if VOLUME_CLOSURE_TARGETS and all(t in idx and active[idx[t]] for t in VOLUME_CLOSURE_TARGETS):
        terms['volume_closure']=((sum(p[:,idx[t]] for t in VOLUME_CLOSURE_TARGETS)-1)**2).mean()
    else: terms['volume_closure']=zero
    return sum(float(LOSS_WEIGHTS.get(k,0))*v for k,v in terms.items()),terms

def stdsafe(prep,j): return max(float(prep['stds'][j]),1e-4)

def sample_weights(df):
    if not USE_WELL_BALANCING: return np.ones(len(df),dtype='float32')
    counts=df['Well'].value_counts()
    w=df['Well'].map(lambda k:1.0/counts[k]).to_numpy(dtype=np.float64,copy=True)
    return (w*len(w)/w.sum()).astype('float32')

print('Device:',DEVICE,'features:',len(FEATURES),'targets:',len(TARGET_LIST))
print('Physics weights:',LOSS_WEIGHTS)


Device: cpu features: 49 targets: 20
Physics weights: {'porosity_order': 0.12, 'fraction_bounds': 0.08, 'permeability_positive': 0.03, 'density_proxy': 0.0, 'archie_proxy': 0.0, 'volume_closure': 0.0}


In [54]:
# Cell 7B — Train with masked targets, internal early stopping and LOWO predictions

def fit_nn(train_df, epochs=NN_EPOCHS, internal_validation=True):
    train_df=train_df.sort_values(['Well','MDKB_m']).reset_index(drop=True)
    if internal_validation:
        tr_ids=[]; va_ids=[]
        for well,grp in train_df.groupby('Well',sort=False):
            ix=grp.index.to_numpy(); split=max(1,int(len(ix)*(1-NN_VALIDATION_FRACTION)))
            tr_ids.extend(ix[:split]); va_ids.extend(ix[split:])
        tr_df=train_df.iloc[tr_ids]; va_df=train_df.iloc[va_ids]
    else:
        tr_df=train_df; va_df=None
    prep,x,y,mask=prepare_training(tr_df)
    if not prep['active'].any():raise ValueError('No target has sufficient labels for training')
    weights=sample_weights(tr_df)
    rawkeys=['PHYS_DENSITY_POROSITY','PHYS_ARCHIE_SW_PROXY']
    # Raw physics proxy tensors are included only if their penalties are enabled.
    raw=np.stack([pd.to_numeric(tr_df[k],errors='coerce').to_numpy(dtype='float32') if k in tr_df else np.full(len(tr_df),np.nan,dtype='float32') for k in rawkeys],axis=1)
    dataset=TensorDataset(torch.from_numpy(x),torch.from_numpy(y),torch.from_numpy(mask),torch.from_numpy(weights),torch.from_numpy(raw))
    loader=DataLoader(dataset,batch_size=NN_BATCH_SIZE,shuffle=True,num_workers=0)
    model=PhysicsNet(len(FEATURES),len(TARGET_LIST)).to(DEVICE)
    optimizer=torch.optim.AdamW(model.parameters(),lr=NN_LR,weight_decay=NN_WEIGHT_DECAY)
    if va_df is not None and len(va_df):
        xv=torch.from_numpy(transform_x(va_df,prep)).to(DEVICE)
        yv=va_df[['FINAL_'+t for t in TARGET_LIST]].apply(pd.to_numeric,errors='coerce').to_numpy(dtype=np.float64,copy=True)
        yv=(yv-prep['means'])/prep['stds']; mv=np.isfinite(yv)&prep['active'][None,:]
        yv=torch.from_numpy(np.where(mv,yv,0).astype('float32')).to(DEVICE)
        mv=torch.from_numpy(mv.astype('float32')).to(DEVICE)
    best_state=None; best_score=np.inf; best_epoch=0; wait=0; history=[]
    for epoch in range(1,int(epochs)+1):
        model.train(); epoch_sum=0.0; nb=0
        for xb,yb,mb,wb,rb in loader:
            xb,yb,mb,wb,rb=[z.to(DEVICE) for z in (xb,yb,mb,wb,rb)]
            optimizer.zero_grad(set_to_none=True)
            pred=model(xb)
            per_row=((pred-yb).square()*mb).sum(dim=1)/mb.sum(dim=1).clamp_min(1)
            supervised=(per_row*wb).sum()/wb.sum().clamp_min(1e-6)
            proxies={k:rb[:,i] for i,k in enumerate(rawkeys)}
            phys,_=physics_loss(pred,prep,proxies)
            loss=supervised+phys
            if not torch.isfinite(loss):raise RuntimeError('Non-finite loss; inspect target scales and physics penalties')
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),NN_GRAD_CLIP)
            optimizer.step(); epoch_sum+=float(loss.detach()); nb+=1
        train_loss=epoch_sum/max(nb,1)
        if va_df is not None and len(va_df):
            model.eval()
            with torch.no_grad():
                pred=model(xv)
                val_loss=float((((pred-yv).square()*mv).sum()/mv.sum().clamp_min(1)).item())
        else: val_loss=train_loss
        history.append({'Epoch':epoch,'Train_Loss':train_loss,'Internal_Val_MSE':val_loss})
        if val_loss<best_score-1e-5:
            best_score=val_loss; best_epoch=epoch; best_state=copy.deepcopy(model.state_dict()); wait=0
        else: wait+=1
        if epoch==1 or epoch%10==0:
            print(f'  epoch {epoch:3d}: train={train_loss:.5f}, internal_val={val_loss:.5f}')
        if internal_validation and wait>=NN_PATIENCE:break
    model.load_state_dict(best_state)
    model.eval()
    return model,prep,pd.DataFrame(history),best_epoch

@torch.no_grad()
def predict_nn(model,prep,df,batch_size=4096):
    model.eval(); x=transform_x(df,prep); parts=[]
    for start in range(0,len(x),batch_size):
        yp=model(torch.from_numpy(x[start:start+batch_size]).to(DEVICE))
        parts.append(yp.cpu().numpy())
    pred=np.concatenate(parts,axis=0)*prep['stds']+prep['means'] if parts else np.empty((0,len(TARGET_LIST)))
    pred[:,~prep['active']]=np.nan
    return pred

CV_RECORDS=[]; OOF_PARTS=[]; FOLD_HISTORY=[]; FOLD_EPOCHS=[]
for holdout in WELLS:
    parts=[]
    for well in WELLS:
        if well==holdout:continue
        frame=JOINED[well]; good=valid_inputs(frame)
        parts.append(select_depth_samples(frame.loc[good].sort_values('MDKB_m'),MAX_TRAIN_ROWS_PER_WELL))
    train=pd.concat(parts,ignore_index=True)
    # Only use rows with at least one observed composite target.
    train=train.loc[train[['FINAL_'+t for t in TARGET_LIST]].notna().any(axis=1)].copy()
    test=JOINED[holdout]; test=test.loc[valid_inputs(test)].sort_values('MDKB_m')
    test=select_depth_samples(test,MAX_QC_ROWS_PER_WELL)
    if len(train)<MIN_TRAIN_SAMPLES or len(test)<MIN_TEST_SAMPLES:
        raise RuntimeError(f'{holdout}: insufficient training/test rows: {len(train)}/{len(test)}')
    print(f'\nLOWO holdout {holdout}: train={len(train):,}, test={len(test):,}')
    model,prep,hist,best_epoch=fit_nn(train)
    FOLD_EPOCHS.append(best_epoch)
    hist['Heldout_Well']=holdout;FOLD_HISTORY.append(hist)
    yp=predict_nn(model,prep,test)
    for j,target in enumerate(TARGET_LIST):
        yt=pd.to_numeric(test['FINAL_'+target],errors='coerce').to_numpy(dtype=float)
        ok=np.isfinite(yt)&np.isfinite(yp[:,j])
        if ok.sum()<MIN_TEST_SAMPLES:
            CV_RECORDS.append({'Target':target,'Heldout_Well':holdout,'Status':'SKIPPED','Test_N':int(ok.sum())})
            continue
        true,pred=yt[ok],yp[ok,j]
        rmse=float(np.sqrt(mean_squared_error(true,pred)))
        spread=float(np.percentile(true,95)-np.percentile(true,5))
        r=float(np.corrcoef(true,pred)[0,1]) if np.std(true)>0 and np.std(pred)>0 else np.nan
        CV_RECORDS.append({'Target':target,'Heldout_Well':holdout,'Status':'EVALUATED','Train_N':len(train),'Test_N':int(ok.sum()),'MAE':float(mean_absolute_error(true,pred)),'RMSE':rmse,'NRMSE':rmse/spread if spread>1e-12 else np.nan,'R2':float(r2_score(true,pred)),'Pearson_r':r,'Best_Epoch':best_epoch})
        o=test.loc[ok,['Well','MDKB_m','TVDSS_m','TWT_ms']].copy()
        o['Target']=target;o['Candidate']='physics_loss_nn';o['Final']=true;o['Predicted']=pred
        OOF_PARTS.append(o)
CV=pd.DataFrame(CV_RECORDS)
OOF=pd.concat(OOF_PARTS,ignore_index=True) if OOF_PARTS else pd.DataFrame(columns=['Well','TWT_ms','Target','Final','Predicted','Candidate'])
chosen={t:{'name':'physics_loss_nn'} for t in TARGET_LIST if t in CV.loc[CV.Status=='EVALUATED','Target'].values}
CV.to_csv(OUT/'PhysicsLossNN_LOWO_Validation.csv',index=False)
OOF.to_csv(OUT/'PhysicsLossNN_LOWO_Predictions.csv',index=False)
pd.concat(FOLD_HISTORY,ignore_index=True).to_csv(OUT/'PhysicsLossNN_Training_History.csv',index=False)
print('\nLOWO validation complete; median best epoch:',int(np.median(FOLD_EPOCHS)))
display(CV)



LOWO holdout Well_4: train=15,221, test=10,364
  epoch   1: train=12.84436, internal_val=0.69611
  epoch  10: train=0.27429, internal_val=0.41887
  epoch  20: train=0.21204, internal_val=0.39563
  epoch  30: train=0.18029, internal_val=0.39751
  epoch  40: train=0.15808, internal_val=0.39817

LOWO holdout Well_6: train=17,583, test=7,802
  epoch   1: train=1.21592, internal_val=1.38777
  epoch  10: train=0.24718, internal_val=0.63527
  epoch  20: train=0.19186, internal_val=0.55001
  epoch  30: train=0.16476, internal_val=0.51829
  epoch  40: train=0.14646, internal_val=0.49550
  epoch  50: train=0.13260, internal_val=0.51746
  epoch  60: train=0.12593, internal_val=0.49793

LOWO holdout Well_7: train=17,966, test=7,419
  epoch   1: train=1.92539, internal_val=1.26655
  epoch  10: train=0.25595, internal_val=0.50234
  epoch  20: train=0.20833, internal_val=0.43250
  epoch  30: train=0.17603, internal_val=0.38681
  epoch  40: train=0.15324, internal_val=0.38168
  epoch  50: train=0.136

,Target,Heldout_Well,Status,Train_N,Test_N,MAE,RMSE,NRMSE,R2,Pearson_r,Best_Epoch
0,FCOAL,Well_4,EVALUATED,15221.0,10007,3.474762e-02,0.138700,NaN,0.171423,0.482896,31.0
1,DTCOM,Well_4,EVALUATED,15221.0,10164,5.873836e+00,8.075205,0.165093,0.687475,0.866900,31.0
2,DTSH,Well_4,EVALUATED,15221.0,9988,1.993561e+01,23.937165,0.150713,0.730789,0.915650,31.0
3,PERMH,Well_4,EVALUATED,15221.0,10007,5.026235e+01,134.699731,10.532814,0.205549,0.527881,31.0
4,PHIE,Well_4,EVALUATED,15221.0,10007,1.649470e-02,0.025970,0.163692,0.751003,0.891911,31.0
5,PHIT,Well_4,EVALUATED,15221.0,10007,2.092283e-02,0.032784,0.156412,0.730143,0.887927,31.0
6,DENB,Well_4,EVALUATED,15221.0,10062,2.802458e-02,0.042943,0.116440,0.884127,0.958658,31.0
7,SWE,Well_4,EVALUATED,15221.0,10007,5.002018e-02,0.080505,0.207317,0.674579,0.870393,31.0
8,SWT,Well_4,EVALUATED,15221.0,10007,3.037281e-02,0.071761,0.194739,0.725125,0.887059,31.0
9,VCLB,Well_4,EVALUATED,15221.0,10007,2.282492e-02,0.029145,0.186947,0.625365,0.863317,31.0


In [55]:
# Cell 8 — Presentation-ready Phase 1 QC: TWT logs + 45-degree crossplots
# Reuses LOWO predictions from Cell 7B; no retraining required.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output

# ─── EDITABLE PRESENTATION SETTINGS ─────────────────────────────────────────
PREFERRED = ['PHIE', 'SWE', 'VGAS', 'VOIL', 'VCLW', 'VSAND', 'VSILT', 'FCOAL', 'PERMH']
PLOT_TARGETS = [t for t in PREFERRED if t in chosen]
FONT_TITLE = 23
FONT_AXIS_LABEL = 19
FONT_TICKS = 16
FONT_MARKER = 14
FONT_LEGEND = 16
FONT_METRIC = 14
PAIRS_PER_ROW = 2             # two (log + crossplot) pairs = four panels per row
TARGETS_PER_PAGE = 4           # two rows per page for readable presentation fonts
FIG_WIDTH = 28                # width of four panels, inches
FIG_HEIGHT_PER_ROW = 8.0      # height per row, inches
EXPORT_SELECTED_PAGE = True   # save selected well/page as PNG
SHOW_FIGURE = True
EXPORT_DPI = 220
MARKER_LINE_ALPHA = 0.72
MAX_MARKER_LABELS = 35
MARKER_LABEL_POSITION = 0.025
CROSSPLOT_MAX_POINTS = 25000   # Downsample displayed points only, not metrics
CROSSPLOT_POINT_SIZE = 10
CROSSPLOT_POINT_ALPHA = 0.24


if not PLOT_TARGETS:
    raise ValueError('No requested composite targets are present in chosen. Run Cell 7B first.')

OUT = Path(OUT)
OUT.mkdir(parents=True, exist_ok=True)

def _well_markers(well_data, tmin, tmax):
    marker_df = well_data.get('markers')
    if marker_df is None or len(marker_df) == 0:
        return pd.DataFrame(columns=['TWT_ms', 'Marker'])
    m = marker_df.copy()
    if 'TWT_ms' not in m.columns:
        return pd.DataFrame(columns=['TWT_ms', 'Marker'])
    m['TWT_ms'] = pd.to_numeric(m['TWT_ms'], errors='coerce')
    if 'Marker' not in m.columns:
        m['Marker'] = 'Marker'
    return m.loc[m['TWT_ms'].between(tmin, tmax)].dropna(subset=['TWT_ms']).sort_values('TWT_ms')

def _target_predictions(well, target):
    candidate = chosen[target]['name']
    cols = ['TWT_ms', 'Final', 'Predicted']
    data = OOF.loc[
        (OOF['Well'] == well) & (OOF['Target'] == target) &
        (OOF['Candidate'] == candidate), cols
    ].copy()
    for col in cols:
        data[col] = pd.to_numeric(data[col], errors='coerce')
    return data.replace([np.inf, -np.inf], np.nan).dropna(subset=cols).sort_values('TWT_ms')

def _metrics(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    if n == 0:
        return dict(n=0, r=np.nan, r2=np.nan, rmse=np.nan, mae=np.nan, bias=np.nan)
    err = y-x
    r = float(np.corrcoef(x,y)[0,1]) if n > 1 and np.std(x)>0 and np.std(y)>0 else np.nan
    # R² is 1 - SSE/SST, not the same as squared correlation.
    sst = np.sum((x-x.mean())**2)
    r2 = float(1-np.sum(err**2)/sst) if n > 1 and sst > 0 else np.nan
    return dict(n=n, r=r, r2=r2, rmse=float(np.sqrt(np.mean(err**2))),
                mae=float(np.mean(np.abs(err))), bias=float(np.mean(err)))

def _plot_property(well, target, well_data, tmin, tmax, ax_log, ax_cross):
    sub = _target_predictions(well, target)
    ax_log.set_title(f'{target} | Well-log comparison', fontsize=FONT_TITLE, weight='bold', pad=15)
    ax_cross.set_title(f'{target} | Prediction crossplot', fontsize=FONT_TITLE, weight='bold', pad=15)
    for ax in (ax_log, ax_cross):
        ax.tick_params(axis='both', labelsize=FONT_TICKS, width=1.2, length=6)
        ax.grid(alpha=.18, linewidth=.8)
    ax_log.set_ylim(tmax, tmin)  # shallow TWT at top
    ax_log.set_ylabel('Two-way time (ms)', fontsize=FONT_AXIS_LABEL, labelpad=12)
    ax_log.set_xlabel(f'{target} (composite units)', fontsize=FONT_AXIS_LABEL, labelpad=10)
    ax_cross.set_xlabel('Finalized composite (log)', fontsize=FONT_AXIS_LABEL, labelpad=10)
    ax_cross.set_ylabel('NN-predicted composite', fontsize=FONT_AXIS_LABEL, labelpad=10)

    if sub.empty:
        for ax in (ax_log, ax_cross):
            ax.text(.5,.5,'No held-out predictions',transform=ax.transAxes,
                    ha='center',va='center',fontsize=FONT_TICKS)
        return None

    # Left: composite curves versus TWT
    ax_log.plot(sub['Final'],sub['TWT_ms'],color='#111827',lw=1.8,
                alpha=.9,zorder=3,label='Finalized composite')
    ax_log.plot(sub['Predicted'],sub['TWT_ms'],color='#D97706',lw=1.8,
                alpha=.9,zorder=4,label='Phase 1 NN prediction')

    # Overlay well markers across the log track; labels sit inside track bounds.
    markers = _well_markers(well_data, tmin, tmax)
    for i,(_, marker) in enumerate(markers.iterrows()):
        y = float(marker['TWT_ms'])
        ax_log.axhline(y,color='#B91C1C',lw=1.0,alpha=MARKER_LINE_ALPHA,zorder=2)
        if i < MAX_MARKER_LABELS:
            ax_log.text(MARKER_LABEL_POSITION,y,str(marker['Marker']),
                        transform=ax_log.get_yaxis_transform(),
                        color='#991B1B',fontsize=FONT_MARKER,weight='bold',
                        va='bottom',ha='left',clip_on=True,
                        bbox=dict(facecolor='white',edgecolor='none',alpha=.76,pad=.5),zorder=6)

    # Right: actual-versus-predicted scatter with identical x/y limits and 1:1 line.
    true = sub['Final'].to_numpy(dtype=float)
    pred = sub['Predicted'].to_numpy(dtype=float)
    metrics = _metrics(true,pred)
    sample_idx = np.linspace(0,len(sub)-1,min(len(sub),CROSSPLOT_MAX_POINTS),dtype=int)
    ax_cross.scatter(true[sample_idx],pred[sample_idx],s=CROSSPLOT_POINT_SIZE,
                     color='#2563EB',alpha=CROSSPLOT_POINT_ALPHA,edgecolors='none',
                     rasterized=True,zorder=3,label='Held-out predictions')
    lo = float(min(np.min(true),np.min(pred)))
    hi = float(max(np.max(true),np.max(pred)))
    span = max(hi-lo, 1e-8)
    margin = span*.05
    lo -= margin
    hi += margin
    ax_cross.plot([lo,hi],[lo,hi],linestyle='--',lw=2.5,color='#DC2626',
                  label='45° identity line (y = x)',zorder=4)
    ax_cross.set_xlim(lo,hi)
    ax_cross.set_ylim(lo,hi)
    ax_cross.set_aspect('equal', adjustable='box')  # truly 45 degrees on screen
    text = (f"N = {metrics['n']:,}\n"
            f"r = {metrics['r']:.3f}    R² = {metrics['r2']:.3f}\n"
            f"RMSE = {metrics['rmse']:.4g}\n"
            f"MAE = {metrics['mae']:.4g}    Bias = {metrics['bias']:.4g}")
    ax_cross.text(.03,.97,text,transform=ax_cross.transAxes,ha='left',va='top',
                  fontsize=FONT_METRIC,weight='bold',
                  bbox=dict(facecolor='white',edgecolor='#CBD5E1',alpha=.93,pad=7),zorder=8)
    ax_cross.legend(loc='lower right',fontsize=FONT_LEGEND-2,framealpha=.9)
    return metrics

# ─── Interactive QC: only the selected well and page are rendered ───────────
# Two side-by-side pairs per row: [Log A | 45° A] [Log B | 45° B]
# Paginate properties so large fonts remain legible in notebook and PowerPoint.

PHASE1_QC_WELLS = [w for w in WELLS if w in JOINED]
if not PHASE1_QC_WELLS:
    raise ValueError('No wells found in both WELLS and JOINED. Run preprocessing cells.')

phase1_qc_well = widgets.Dropdown(
    options=PHASE1_QC_WELLS, value=PHASE1_QC_WELLS[0],
    description='Well:', style={'description_width': 'initial'},
    layout=widgets.Layout(width='280px')
)
phase1_qc_pages = [PLOT_TARGETS[i:i+TARGETS_PER_PAGE]
                   for i in range(0, len(PLOT_TARGETS), TARGETS_PER_PAGE)]
phase1_qc_page = widgets.Dropdown(
    options=[(f'Page {i+1}: {", ".join(p)}', i)
             for i, p in enumerate(phase1_qc_pages)],
    value=0, description='Properties:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='480px')
)
phase1_qc_export = widgets.Checkbox(
    value=EXPORT_SELECTED_PAGE, description='Save selected page as PNG',
    indent=False, layout=widgets.Layout(width='230px')
)
phase1_qc_redraw_button = widgets.Button(description='Refresh plot', icon='refresh')
phase1_qc_output = widgets.Output()
phase1_presentation_metrics = pd.DataFrame()

def phase1_qc_render(_=None):
    global phase1_presentation_metrics
    well = phase1_qc_well.value
    well_data = WELLS[well]
    targets = phase1_qc_pages[phase1_qc_page.value]

    with phase1_qc_output:
        clear_output(wait=True)
        times = pd.to_numeric(JOINED[well]['TWT_ms'], errors='coerce').to_numpy(dtype=float)
        times = times[np.isfinite(times)]
        if not len(times):
            print(f'{well}: no finite TWT values available.')
            return
        tmin, tmax = float(times.min()), float(times.max())
        all_markers = _well_markers(well_data, -np.inf, np.inf)
        if len(all_markers):
            tmin = min(tmin, float(all_markers['TWT_ms'].min()))
            tmax = max(tmax, float(all_markers['TWT_ms'].max()))
        pad = max(5.0, (tmax - tmin) * 0.025)
        tmin -= pad
        tmax += pad

        nrows = (len(targets) + PAIRS_PER_ROW - 1) // PAIRS_PER_ROW
        fig, axes = plt.subplots(
            nrows, 2 * PAIRS_PER_ROW,
            figsize=(FIG_WIDTH, FIG_HEIGHT_PER_ROW * nrows),
            squeeze=False,
            gridspec_kw={'width_ratios': [1.0, 1.12] * PAIRS_PER_ROW},
            constrained_layout=True
        )
        summary = []
        for i, target in enumerate(targets):
            row, pair = divmod(i, PAIRS_PER_ROW)
            ax_log, ax_cross = axes[row, 2 * pair:2 * pair + 2]
            stats = _plot_property(well, target, well_data, tmin, tmax, ax_log, ax_cross)
            if stats is not None:
                summary.append({'Well': well, 'Target': target, **stats})
        for i in range(len(targets), nrows * PAIRS_PER_ROW):
            row, pair = divmod(i, PAIRS_PER_ROW)
            axes[row, 2 * pair].set_visible(False)
            axes[row, 2 * pair + 1].set_visible(False)

        handles = [
            Line2D([0], [0], color='#111827', lw=3, label='Finalized composite'),
            Line2D([0], [0], color='#D97706', lw=3, label='NN prediction'),
            Line2D([0], [0], color='#B91C1C', lw=2, label='Geological marker'),
        ]
        fig.legend(handles=handles, loc='upper center',
                   bbox_to_anchor=(.5, 1.045), ncol=3,
                   fontsize=FONT_LEGEND, frameon=False)
        fig.suptitle(f'{well} — Phase 1 LOWO QC | Page {phase1_qc_page.value + 1}',
                     fontsize=FONT_TITLE + 3, weight='bold', y=1.075)

        if phase1_qc_export.value:
            filename = f'{well}_Phase1_Presentation_QC_Page{phase1_qc_page.value+1}_TwoPairsPerRow.png'
            path = OUT / filename
            fig.savefig(path, dpi=EXPORT_DPI, bbox_inches='tight', facecolor='white')
            print('Saved presentation PNG:', path)
        if SHOW_FIGURE:
            display(fig)
        plt.close(fig)
        if summary:
            phase1_presentation_metrics = pd.DataFrame(summary)
            display(phase1_presentation_metrics)

phase1_qc_redraw_button.on_click(phase1_qc_render)
phase1_qc_well.observe(phase1_qc_render, names='value')
phase1_qc_page.observe(phase1_qc_render, names='value')
display(widgets.VBox([
    widgets.HBox([phase1_qc_well, phase1_qc_page]),
    widgets.HBox([phase1_qc_export, phase1_qc_redraw_button]),
    phase1_qc_output
]))
phase1_qc_render()


## Cell 9 — Final model

After LOWO diagnostics, fit one model on all three wells. The training duration uses the **median internal best epoch** from the three LOWO folds; no held-out labels are used for hyperparameter selection within a fold. Note that this is still model-selection evidence, not an untouched fourth-well test.

In [56]:
# Cell 9 — Final model fit on all three wells
train_parts=[]
for well in WELLS:
    df=JOINED[well]; good=valid_inputs(df)
    train_parts.append(select_depth_samples(df.loc[good].sort_values('MDKB_m'),MAX_FINAL_ROWS_PER_WELL))
FINAL_TRAIN=pd.concat(train_parts,ignore_index=True)
FINAL_TRAIN=FINAL_TRAIN.loc[FINAL_TRAIN[['FINAL_'+t for t in TARGET_LIST]].notna().any(axis=1)].copy()
FINAL_EPOCHS=max(5,int(np.median(FOLD_EPOCHS)))
print('Final training rows:',len(FINAL_TRAIN),'epochs:',FINAL_EPOCHS)
FINAL_MODEL,FINAL_PREP,FINAL_HISTORY,_=fit_nn(FINAL_TRAIN,epochs=FINAL_EPOCHS,internal_validation=False)
FINAL_HISTORY.to_csv(OUT/'PhysicsLossNN_Final_Training_History.csv',index=False)
print('Final model trained. Active targets:',[t for t,a in zip(TARGET_LIST,FINAL_PREP['active']) if a])


Final training rows: 25385 epochs: 52
  epoch   1: train=1.19511, internal_val=1.19511
  epoch  10: train=0.23237, internal_val=0.23237
  epoch  20: train=0.18326, internal_val=0.18326
  epoch  30: train=0.15515, internal_val=0.15515
  epoch  40: train=0.13728, internal_val=0.13728
  epoch  50: train=0.12812, internal_val=0.12812
Final model trained. Active targets: ['FCOAL', 'DTCOM', 'DTSH', 'PERMH', 'PHIE', 'PHIT', 'DENB', 'SWE', 'SWT', 'VCLB', 'VCLD', 'VCLW', 'VDOLO', 'VGAS', 'VLIME', 'VOIL', 'VSAND', 'VSHALE', 'VSHGR', 'VSILT']


In [57]:
# Cell 10 — Save PyTorch weights, preprocessing and manifest
# IMPORTANT: Only load serialized files you trust.
MODEL_PATH=OUT/'Well4_6_7_PhysicsLossNN.pt'
PREP_PATH=OUT/'Well4_6_7_PhysicsLossNN_Preprocessing.joblib'
torch.save({'state_dict':{k:v.detach().cpu() for k,v in FINAL_MODEL.state_dict().items()},
            'n_features':len(FEATURES),'n_targets':len(TARGET_LIST),
            'hidden':list(NN_HIDDEN),'dropout':NN_DROPOUT},MODEL_PATH)
joblib.dump(FINAL_PREP,PREP_PATH,compress=3)
MANIFEST={
    'model':'PyTorch physics-loss multi-output regression','training_wells':list(WELLS),
    'features':FEATURES,'inputs':INPUTS,'targets':TARGET_LIST,
    'active_targets':[t for t,a in zip(TARGET_LIST,FINAL_PREP['active']) if a],
    'marker_vocabulary':MARKER_VOCABULARY,'marker_features_enabled':USE_MARKER_FEATURES,
    'marker_features':MARKER_FEATURES if USE_MARKER_FEATURES else [],
    'approved_marker_mapping':APPROVED_MARKER_MAPPING,
    'physics_features_enabled':USE_PHYSICS_FEATURES,'physics_features':PHYSICS_FEATURES if USE_PHYSICS_FEATURES else [],
    'physics_params':PHYSICS_PARAMS,'loss_weights':LOSS_WEIGHTS,
    'volume_closure_targets':VOLUME_CLOSURE_TARGETS,
    'datum_offsets_m':DATUM_OVERRIDES_M,'hidden':list(NN_HIDDEN),'dropout':NN_DROPOUT,
    'epochs':FINAL_EPOCHS,'device_trained':str(DEVICE),
    'validation':'Leave one WELL out, internal blocked early stopping only on training wells',
    'caution':'Provisional physics; Archie and density loss OFF until calibration; independent fourth well needed'
}
(OUT/'PhysicsLossNN_Manifest.json').write_text(json.dumps(MANIFEST,indent=2))
print('Saved:',MODEL_PATH,PREP_PATH,OUT/'PhysicsLossNN_Manifest.json',sep='\n')


Saved:
/home/apgce_team_08/link-to-apgce_team_08/Well_4_6_7_Phase1_Multiwell_CPU_Output/Well4_6_7_PhysicsLossNN.pt
/home/apgce_team_08/link-to-apgce_team_08/Well_4_6_7_Phase1_Multiwell_CPU_Output/Well4_6_7_PhysicsLossNN_Preprocessing.joblib
/home/apgce_team_08/link-to-apgce_team_08/Well_4_6_7_Phase1_Multiwell_CPU_Output/PhysicsLossNN_Manifest.json


In [58]:
# Cell 11 — Apply trained network to a future well (with measured curves and TWT marker picks)
# Use the same datum/checkshot and measured-curve units as training.
def predict_composite_other_well(measured_df, markers_df=None, model_path=MODEL_PATH, prep_path=PREP_PATH):
    # Reuse this notebook's marker_features and physics_features definitions.
    # Only load model files created by you or another trusted source.
    d=measured_df.copy();d.columns=[str(c).strip().upper() for c in d.columns]
    xx=pd.DataFrame(index=d.index)
    for c in INPUTS:
        xx['RAW_'+c]=pd.to_numeric(d[c],errors='coerce') if c in d else np.nan
    xx=xx.replace([np.inf,-np.inf,-9999,-999.25,9999],np.nan)
    valid=xx.notna().any(axis=1)
    out=pd.DataFrame(index=d.index)
    if 'MDKB_M' in d:out['MDKB_m']=d['MDKB_M']
    if 'TWT_MS' in d:out['TWT_ms']=d['TWT_MS']
    out['Input_Curve_Count']=xx.notna().sum(axis=1)
    if USE_MARKER_FEATURES:
        if markers_df is None or 'TWT_MS' not in d:
            raise ValueError('Marker-aware inference requires markers_df with TWT_ms and measured_df with TWT_ms')
        extra=marker_features(pd.to_numeric(d['TWT_MS'],errors='coerce').to_numpy(),markers_df,MARKER_VOCABULARY)
        for c in MARKER_FEATURES:xx[c]=extra[c].to_numpy()
    if USE_PHYSICS_FEATURES:
        extra=physics_features(xx,PHYSICS_PARAMS)
        for c in PHYSICS_FEATURES:xx[c]=extra[c].to_numpy()
    prep=joblib.load(prep_path)
    try: ckpt=torch.load(model_path,map_location=DEVICE,weights_only=True)
    except TypeError: ckpt=torch.load(model_path,map_location=DEVICE)
    model=PhysicsNet(ckpt['n_features'],ckpt['n_targets'],tuple(ckpt['hidden']),ckpt['dropout']).to(DEVICE)
    model.load_state_dict(ckpt['state_dict']);model.eval()
    if valid.any():
        pred=predict_nn(model,prep,xx.loc[valid,FEATURES])
    for j,t in enumerate(TARGET_LIST):
        out[t+'_ML']=np.nan
        if valid.any():out.loc[valid,t+'_ML']=pred[:,j]
    # Optional physical bounds are post-processing, not proof of improved accuracy.
    if APPLY_PHYSICAL_BOUNDS_AT_INFERENCE:
        for t in FRACTION_TARGETS:
            if t+'_ML' in out:out[t+'_ML']=out[t+'_ML'].clip(0,1)
        if 'PHIE_ML' in out and 'PHIT_ML' in out:
            out['PHIE_ML']=np.minimum(out['PHIE_ML'],out['PHIT_ML'])
        if 'PERMH_ML' in out:out['PERMH_ML']=out['PERMH_ML'].clip(lower=0)
    return out
print('Prediction helper ready; requires same marker/physics feature settings as training.')


Prediction helper ready; requires same marker/physics feature settings as training.


## Interpretation and safeguards

- **Physics-loss network:** The porosity hierarchy, physical fraction bounds and nonnegative permeability are differentiable soft penalties during training. Unlike the earlier tree-based model, the training objective directly includes physics terms.
- **Archie and density proxies:** Their loss weights default to zero. These assumptions are not universally valid in shale, coal or gas-bearing intervals. Do not enable Archie loss without a clean-formation mask, water-resistivity calibration and verification of saturation definition. The optional Archie loss in this prototype is intentionally **not production-ready**.
- **Volume closure:** Disabled until the constituent mineral-volume curves are proven disjoint and exhaustive. Several clay-volume curves may be alternative interpretations.
- **Marker review:** Cell 6C flags similar marker names without automatically merging them. Only user-approved aliases in `APPROVED_MARKER_MAPPING` are merged before creating identity features.
- **Validation:** Missing target labels are masked. LOWO evaluates genuine held-out wells, but three wells remain a small sample and the physics weights are provisional. A fourth well is the recommended final test.
- **Future well:** Use identical measured-curve definitions, units, marker names and TWT conversion. The manifest records the training configuration. Inference in this notebook uses the current notebook's settings; load the manifest and confirm they match before predicting with a saved checkpoint.